# Valid Anagram (LeetCode 242)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OlixIgnacious/writteninchalk-dsa/blob/main/02-frequency-map/valid-anagram.ipynb)

Run the cells in order. Below each solution are cells that call it on an example: change the inputs and run them again.


## The problem


You are given two strings `s` and `t`. Return `true` if `t` is an anagram of `s`, and `false` otherwise. An anagram uses exactly the same letters, each the same number of times, in any order.

| Input | Output | Why |
|---|---|---|
| `s = "anagram"`, `t = "nagaram"` | `true` | Both have three a’s and one each of n, g, r, m |
| `s = "rat"`, `t = "car"` | `false` | `t` has a c; `s` has a t |
| `s = "aacc"`, `t = "ccac"` | `false` | Same letters, different counts: two a’s vs one |

Input limits: 1 ≤ length of each string ≤ 50,000. Both contain only lowercase English letters. LeetCode’s own follow-up: what if the strings contain any Unicode characters?

Ask before you code: Only lowercase a–z? Does case matter? Do spaces and punctuation count?

## Solution 1: brute force


Brute force: for each letter of `s`, find a matching letter in `t` and cross it out.

In [1]:
def is_anagram_brute(s, t):                   # O(n²) time
    if len(s) != len(t):
        return False
    remaining = list(t)
    for ch in s:
        if ch not in remaining:               # t has no unused copy of this letter
            return False
        remaining.remove(ch)                  # cross it out so it can't be used twice
    return True

Why it's slow: `in` and `remove` on a list both search it from the start, so each letter of `s` can cost up to n steps. n letters, each costing up to n steps, is O(n²).

In [2]:
is_anagram_brute("anagram", "nagaram")

True

In [3]:
is_anagram_brute("rat", "car")

False

In [4]:
is_anagram_brute("aacc", "ccac")

False

## Solution 2: sort both


Sort both: anagrams look identical once sorted.

In [5]:
def is_anagram_sort(s, t):                    # O(n log n) time
    return sorted(s) == sorted(t)

Why sorting is O(n log n): Python's `sorted` uses Timsort, a refined merge sort. A run is a stretch of the list that is already in order, like `1, 2, 5` in `[1, 2, 5, 3, 4]`. Timsort finds the runs, makes short ones longer by sorting small pieces, then merges them the way merge sort does. An already-sorted list is one run, so it takes about n steps; on text with no order in it the runs are short, and the cost is the same as merge sort's. Merge sort shows where that cost comes from. It splits the list in half, then in half again, until every piece is a single item. That takes about log₂ n rounds of halving: 10 rounds for 1,000 items, about 16 for 50,000. It then merges the pieces back together in order, and each round of merging touches all n items once. n items in each of log n rounds is O(n log n): far less than brute force's O(n²), but still more than one pass.

In [6]:
sorted("anagram")

['a', 'a', 'a', 'g', 'm', 'n', 'r']

In [7]:
sorted("nagaram")

['a', 'a', 'a', 'g', 'm', 'n', 'r']

In [8]:
is_anagram_sort("anagram", "nagaram")

True

In [9]:
is_anagram_sort("rat", "car")

False

In [10]:
is_anagram_sort("aacc", "ccac")

False

## Solution 3: frequency map


Frequency map (the best answer, and the one to present in the interview):

Build a frequency map: count every letter in `s`. Then go through `t` and subtract one for each letter. If a letter’s count would drop below zero, `t` uses that letter more often than `s` does, so return `false`.

Check the lengths first. Strings of different lengths can never be anagrams. And when the lengths are equal, “no count ever went below zero” means every count ended at exactly zero, so you don’t need a final check. Here's why. `s` adds one for each of its n letters and `t` takes one away for each of its n letters, so the counts always add up to zero at the end. No count can go below zero, because the loop returns `false` as soon as `t` asks for a letter whose count is already zero. Counts that add up to zero, with none below zero, must all be exactly zero.

In [11]:
def is_anagram(s, t):                         # O(n) time
    if len(s) != len(t):                      # different lengths: can't be anagrams
        return False
    count = {}
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
    for ch in t:
        if count.get(ch, 0) == 0:             # t needs a letter s has run out of
            return False
        count[ch] -= 1
    return True

Why `count.get(ch, 0)` and not `count[ch]`: reading a key the dict doesn't have, like `count['c']` when `s` has no c, raises a `KeyError`. `get` returns 0 instead. The subtraction can use `count[ch]`, because the `if` above it has just made sure that `ch`'s count isn't zero.

Why is a lookup one step? A `dict` or `set` is a hash table. Python runs `hash(x)` to turn the value into a number, and that number picks a slot directly. So `x in seen` jumps to one slot instead of scanning the whole collection. A list has no such shortcut: `x in some_list` checks every element.

Is it always one step? No. It’s one step on average. If many values land in the same slot (a collision), Python has to check each of them, and in the worst case a lookup becomes O(n). This is rare in practice; Python also randomises how strings are hashed so it can’t be triggered on purpose. In an interview, say “O(1) average, O(n) worst case”.

### Walkthrough: s = "anagram", t = "nagaram"

After counting `s`: a = 3, n = 1, g = 1, r = 1, m = 1.

| Letter from `t` | Count before | Count after | Below zero? |
|---|---|---|---|
| n | 1 | 0 | No |
| a | 3 | 2 | No |
| g | 1 | 0 | No |
| a | 2 | 1 | No |
| r | 1 | 0 | No |
| a | 1 | 0 | No |
| m | 1 | 0 | No |

Nothing went below zero, so the answer is `true`. For `s = "rat"`, `t = "car"`, the very first letter of `t` is c, whose count is 0, so it returns `false` immediately.

In [12]:
is_anagram("anagram", "nagaram")

True

In [13]:
is_anagram("rat", "car")

False

In [14]:
is_anagram("aacc", "ccac")

False

## Solution 4: a list of 26 counts


A list of 26 counts: only for lowercase a–z. `ord(ch) - ord('a')` turns ‘a’ into 0, ‘b’ into 1, up to ‘z’ into 25, so each letter gets its own slot.

In [15]:
def is_anagram_array(s, t):                   # O(n) time, O(1) memory
    if len(s) != len(t):
        return False
    count = [0] * 26
    for a, b in zip(s, t):
        count[ord(a) - ord('a')] += 1         # letter from s: add
        count[ord(b) - ord('a')] -= 1         # letter from t: subtract
    return all(c == 0 for c in count)

Why it's the fastest: a list slot is found straight from its index, with no hashing, and one loop walks both strings together. Why it breaks on other text: only a–z give an index from 0 to 25. A capital 'Z' gives `ord('Z') - ord('a')`, which is -7, and a negative index counts from the end of the list, so -7 is slot 19, the slot for 't'. That is the wrong slot, and no error. An 'é' gives 136, past the end of the list, so Python raises an `IndexError`. Why it needs a final check: it adds for `s` and subtracts for `t` in the same pass, so a slot can dip below zero partway through and still end at zero. With "ab" and "ba", slot b is -1 after the first pair and 0 at the end. So it can't return early: only when both strings are done do you know that every slot is back at zero. The dict version counts all of `s` first, so a zero count in the `t` loop really means `t` has run out.

In [16]:
is_anagram_array("anagram", "nagaram")

True

In [17]:
is_anagram_array("rat", "car")

False

In [18]:
is_anagram_array("aacc", "ccac")

False

In [19]:
ord("Z") - ord("a")

-7

In [20]:
is_anagram_array("Z", "t")

True

In [21]:
try:
    is_anagram_array("é", "e")
except IndexError as error:
    print("IndexError:", error)

IndexError: list index out of range


In [22]:
is_anagram("Z", "t")

False

## Solution 5: Counter


Standard library: fine in interviews, but be ready to write version 3. An interviewer may ask you to count without `Counter`, because `Counter` hides the counting they want to see you do.

In [23]:
from collections import Counter

def is_anagram_counter(s, t):
    return Counter(s) == Counter(t)

In [24]:
Counter("anagram")

Counter({'a': 3, 'n': 1, 'g': 1, 'r': 1, 'm': 1})

In [25]:
is_anagram_counter("anagram", "nagaram")

True

In [26]:
is_anagram_counter("rat", "car")

False

In [27]:
is_anagram_counter("aacc", "ccac")

False

## All five compared


| Approach | Time | Extra memory | Note |
|---|---|---|---|
| Brute force | O(n²) | O(n), a copy of `t` | Too slow for long strings |
| Sort both | O(n log n) | O(n) | Simple starting point |
| Frequency map | O(n) | One entry per distinct letter | Default answer; works for any characters (normalise Unicode text first) |
| 26 counts | O(n) | O(1), always 26 slots | Fastest, lowercase a–z only |
| `Counter` | O(n) | One entry per distinct letter | Shortest to write |

## Similar problems to try


Each one is a frequency map at heart: count first, then answer from the counts. Try them yourself before looking anything up.

| Problem | What it asks |
|---|---|
| Ransom Note (LeetCode 383) | Can a note be built from the letters of a magazine, using each letter at most once? |
| Minimum Number of Steps to Make Two Strings Anagram (LeetCode 1347) | How many letters of `t` must change to make it an anagram of `s`? |
| Unique Number of Occurrences (LeetCode 1207) | Does every value in the array appear a different number of times? |
| First Unique Character in a String (LeetCode 387) | Which is the first character in a string that doesn't repeat? Give its index, or -1 if there is none. |